# 03 — GAN + Channel Attention

Adds **channel attention** at the bottleneck (CBAM-style shared MLP over global avg/max pooling), letting the network re-weight *which feature channels* matter. `build_generator(use_channel=True, use_spatial=False)`.

### Running on Google Colab

```python
# !git clone https://github.com/maurofalc/hdr-imaging.git
# %cd hdr-imaging
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# DATA_ROOT = "/content/drive/MyDrive/SICE"   # set this in the config cell below
```

In [ ]:
import os, sys
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Make the shared package importable (works from the repo root or on Colab after %cd).
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
from hdr_gan import (
    build_generator, build_discriminator, load_sice,
    generator_loss, discriminator_loss, psnr, ssim,
)

In [ ]:
# --- Configuration (shared across every ablation notebook) ---
DATA_ROOT     = r"../hdr-tcc"   # folder that contains Part1/ and Part2/ (SICE)
IMG_SIZE      = 512
EPOCHS        = 50             # the thesis used 100+; keep small for quick runs
BATCH_SIZE    = 8
LEARNING_RATE = 1e-4
SEED          = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Loads SICE with the shared held-out test split (no train/test leakage).
(x_train, y_train), (x_test, y_test) = load_sice(DATA_ROOT, size=IMG_SIZE)
print("train:", x_train.shape, " test:", x_test.shape)

In [ ]:
gen  = build_generator(use_channel=True, use_spatial=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
disc = build_discriminator(input_shape=(IMG_SIZE, IMG_SIZE, 3))
gen_opt  = keras.optimizers.Adam(LEARNING_RATE)
disc_opt = keras.optimizers.Adam(LEARNING_RATE)
gen.summary()

In [ ]:
@tf.function
def train_step(ldr, hdr):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated = gen(ldr, training=True)
        real_output = disc(hdr, training=True)
        fake_output = disc(generated, training=True)
        g_loss = generator_loss(fake_output, hdr, generated)   # content MSE + 1e-3 * adversarial
        d_loss = discriminator_loss(real_output, fake_output)
    gen_opt.apply_gradients(
        zip(gen_tape.gradient(g_loss, gen.trainable_variables), gen.trainable_variables))
    disc_opt.apply_gradients(
        zip(disc_tape.gradient(d_loss, disc.trainable_variables), disc.trainable_variables))
    return g_loss, d_loss

In [ ]:
n_train = x_train.shape[0]
steps = n_train // BATCH_SIZE

for epoch in range(EPOCHS):
    order = np.random.permutation(n_train)
    g_running = d_running = 0.0
    for s in range(steps):
        batch = order[s * BATCH_SIZE:(s + 1) * BATCH_SIZE]
        g_loss, d_loss = train_step(x_train[batch], y_train[batch])
        g_running += float(g_loss); d_running += float(d_loss)
    test_psnr = float(psnr(y_test, gen(x_test, training=False)))
    print(f"epoch {epoch + 1:>3}/{EPOCHS}  "
          f"G={g_running / steps:.4f}  D={d_running / steps:.4f}  test PSNR={test_psnr:.2f}")

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
gen.save_weights(os.path.join("checkpoints", "03_gan_channel.weights.h5"))
print("saved checkpoints/03_gan_channel.weights.h5")

In [ ]:
# Reconstructions on the held-out test set.
preds = gen.predict(x_test, batch_size=BATCH_SIZE)
k = min(4, x_test.shape[0])
fig, ax = plt.subplots(k, 3, figsize=(12, 4 * k))
for i in range(k):
    for col, (img, title) in enumerate([
        (x_test[i], "LDR input"), (preds[i], "Generated HDR"), (y_test[i], "Ground truth")
    ]):
        ax[i, col].imshow(np.clip(img, 0, 1)); ax[i, col].set_title(title); ax[i, col].axis("off")
plt.tight_layout(); plt.show()

print(f"Held-out PSNR: {float(psnr(y_test, preds)):.3f}  SSIM: {float(ssim(y_test, preds)):.3f}")